<a href="https://colab.research.google.com/github/siya-pathak/graphrag-research/blob/main/graphrag_louvain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama-index graspologic numpy==1.24.4 scipy==1.12.0 python-louvain

**Load Data**
1. sample news article dataset
2. 2,500 samples; for ease of experimentation, we will use 5 of these samples, which include the title and text of news articles.

In [ ]:
import pandas as pd
from llama_index.core import Document

news = pd.read_csv(
    "https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/news_articles.csv"
)[:5]

news.head()

,title,date,text
0,Chevron: Best Of Breed,2031-04-06T01:36:32.000000000+00:00,JHVEPhoto Like many companies in the O&G secto...
1,FirstEnergy (NYSE:FE) Posts Earnings Results,2030-04-29T06:55:28.000000000+00:00,FirstEnergy (NYSE:FE – Get Rating) posted its ...
2,Dáil almost suspended after Sinn Féin TD put p...,2023-06-15T14:32:11.000000000+00:00,The Dáil was almost suspended on Thursday afte...
3,Epic’s latest tool can animate hyperrealistic ...,2023-06-15T14:00:00.000000000+00:00,"Today, Epic is releasing a new tool designed t..."
4,"EU to Ban Huawei, ZTE from Internal Commission...",2023-06-15T13:50:00.000000000+00:00,The European Commission is planning to ban equ...


**Document Object:** Each piece of information is encapsulated in a Document object that contains metadata, relationships, and the main content. The object has the following fields:

**id_:** A unique identifier for the document (e.g., 'a8bbf27f-e764-488d-89d6-36dd92bedac9'). This helps in tracking and referencing specific documents.

**embedding:** Indicates any associated embeddings for semantic analysis (currently None, meaning embeddings are not generated or included).

**metadata:** An empty dictionary {} representing metadata associated with the document, which can store information such as the source, author, or publication date.

**excluded_embed_metadata_keys:** A list of metadata keys that should be excluded from embedding operations. It’s empty [], suggesting that all metadata (if present) would be eligible for embedding.

**excluded_llm_metadata_keys:** A list of metadata keys to be excluded from processing by large language models (also empty []).

**relationships:** Represents any relational connections to other documents or entities (currently {}, meaning no relationships are defined).

**text:** The main content of the document, which can vary in length and detail. It’s formatted as a plain string.

**mimetype:** Specifies the type of content, which is 'text/plain' in all cases, indicating simple textual data.

**start_char_idx and end_char_idx:** Indices marking the start and end of the text content (currently None, suggesting that the entire text is used without specific substring indexing).

**text_template:** A template string indicating how to format text with metadata (e.g., '{metadata_str}\n\n{content}'). This field defines how the document content should be presented, with metadata followed by the content.

**metadata_template:** Describes how to format individual metadata items. It uses the pattern '{key}: {value}'.

**metadata_seperator:** Specifies the separator to use between metadata items, which is a newline character '\n'.


In [ ]:
documents = [
    Document(text=f"{row['title']}: {row['text']}")
    for i, row in news.iterrows()
]

print(documents)

[Document(id_='dc999b31-23e2-4eb7-9583-9493b76210b2', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Chevron: Best Of Breed: JHVEPhoto Like many companies in the O&G sector, the stock of Chevron (NYSE:CVX) has declined about 10% over the past 90-days despite the fact that Q2 consensus earnings estimates have risen sharply (~25%) during that same time frame. Over the years, Chevron has kept a very strong balance sheet. That allowed the...', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), Document(id_='e7add900-2a16-477a-9968-5b149d571619', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='FirstEnergy (NYSE:FE) Posts Earnings Results: FirstEnergy (N

**Setup API Key and LLM**

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4")

1. **GraphRAGExtractor** is a class derived from **TransformComponent,** which is part of a **framework for knowledge graph extraction**.
The class extracts **triples (subject-relation-object)** from text using a language model (LLM). It can also **process and add descriptions for entities(subject and object) and relationships**.

Why is metadata.copy() Called Twice?
metadata = node.metadata.copy() is called twice to manage metadata for two different purposes:
First Call (For Entity Nodes): The metadata is copied before iterating over entities. This allows you to modify the **metadata specifically for entity nodes by adding "entity_description"** to it. The modified metadata is then used to **create EntityNode objects.**
Second Call (For Relationship Nodes): The metadata is copied again before iterating over entities_relationship. This **separate copy is modified for relationships, where "relationship_description" is added**. This ensures that any changes made to the metadata for entities do not affect the metadata used for relationships.

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

from typing import Any, List, Callable, Optional, Union, Dict
from IPython.display import Markdown, display

from llama_index.core.async_utils import run_jobs
from llama_index.core.indices.property_graph.utils import (
    default_parse_triplets_fn,
)
from llama_index.core.graph_stores.types import (
    EntityNode,
    KG_NODES_KEY,
    KG_RELATIONS_KEY,
    Relation,
)
from llama_index.core.llms.llm import LLM
from llama_index.core.prompts import PromptTemplate
from llama_index.core.prompts.default_prompts import (
    DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
)
from llama_index.core.schema import TransformComponent, BaseNode
from llama_index.core.bridge.pydantic import BaseModel, Field

class GraphRAGExtractor(TransformComponent):
    """Extract triples from a graph.

    Uses an LLM and a simple prompt + output parsing to extract paths (i.e. triples) and entity, relation descriptions from text.

    Args:
        llm (LLM):
            The language model to use.
        extract_prompt (Union[str, PromptTemplate]):
            The prompt to use for extracting triples.
        parse_fn (callable):
            A function to parse the output of the language model.
        num_workers (int):
            The number of workers to use for parallel processing.
        max_paths_per_chunk (int):
            The maximum number of paths to extract per chunk.
    """

    llm: LLM
    extract_prompt: PromptTemplate
    parse_fn: Callable
    num_workers: int #it determines how many pieces of text can be processed in parallel, which helps speed up the overall extraction process.
    max_paths_per_chunk: int #max_paths_per_chunk refers to the maximum number of triples that the system should extract from a single chunk of text.

    def __init__(
        self,
        llm: Optional[LLM] = None, #if we don't have an llm provided then the value is none
        extract_prompt: Optional[Union[str, PromptTemplate]] = None, #string or prompttemplate is given, string(if given) is converted to prompttemplate
        parse_fn: Callable = default_parse_triplets_fn,
        max_paths_per_chunk: int = 10,
        num_workers: int = 4,
    ) -> None:
        """Init params."""
        from llama_index.core import Settings

        if isinstance(extract_prompt, str): #If a string is passed as the extraction prompt, it converts it into a PromptTemplate object.
            extract_prompt = PromptTemplate(extract_prompt)

        super().__init__(
            llm=llm or Settings.llm, #if llm value is none default llm from settings is used
            extract_prompt=extract_prompt or DEFAULT_KG_TRIPLET_EXTRACT_PROMPT, #if no extract prompt given then default_kg_triplet_extract_prompt used
            parse_fn=parse_fn,
            num_workers=num_workers,
            max_paths_per_chunk=max_paths_per_chunk,
        )

    @classmethod
    def class_name(cls) -> str:
        return "GraphExtractor"

    def __call__(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]: #basenode is part of llama index, specific type of node this is
        """Extract triples from nodes."""
        return asyncio.run(
            self.acall(nodes, show_progress=show_progress, **kwargs)
        )

    async def _aextract(self, node: BaseNode) -> BaseNode: #asynchronous method that extracts triples from a BaseNode object,
    #The function returns a modified BaseNode object with extracted entities and relationships stored in its metadata.
        """Extract triples from a node."""
        assert hasattr(node, "text") #Ensures the node object has a text attribute. If it doesn’t, the function raises an AssertionError.

        text = node.get_content(metadata_mode="llm") #extract its content in a format suitable for language model processing.

        print("Text from node:", text)
        print("Initial metadata:", node.metadata)

        try:
            llm_response = await self.llm.apredict(
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )
            entities, entities_relationship = self.parse_fn(llm_response)
            print("Extracted Entities:", entities)
            print("Extracted Relationships:", entities_relationship)
        except ValueError:
            entities = []
            entities_relationship = []

        #Retrieves and removes existing entity nodes and relationships from node.metadata. If these keys don't exist, empty lists are returned.
        existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        print("Existing KG Nodes:", existing_nodes)
        print("Existing KG Relations:", existing_relations)

        #Makes a copy of node.metadata to be used when creating new nodes and relations.
        metadata = node.metadata.copy()
        print("Metadata copied for entity modification:", metadata)
        for entity, entity_type, description in entities:
            metadata[
                "entity_description"
            ] = description  # Not used in the current implementation. But will be useful in future work.
            entity_node = EntityNode(
                name=entity, label=entity_type, properties=metadata
            )
            existing_nodes.append(entity_node)

        # Metadata after entity modification
        print("Metadata after entity modification:", metadata)
        print("KG Nodes after adding entities:", existing_nodes)

        metadata = node.metadata.copy() #why call second time? this time we copy node's metadata to modify it accoridng to our requirements for relationship specifically, previously metadata was modified for entity
        print("Metadata copied for relationship modification:", metadata)
        for triple in entities_relationship:
            subj, rel, obj, description = triple
            subj_node = EntityNode(name=subj, properties=metadata)
            obj_node = EntityNode(name=obj, properties=metadata)
            metadata["relationship_description"] = description
            rel_node = Relation(
                label=rel,
                source_id=subj_node.id,
                target_id=obj_node.id,
                properties=metadata,
            )

            existing_nodes.extend([subj_node, obj_node]) #same as append(append adds 1 element, extend adds more than 1)
            existing_relations.append(rel_node)

        print("Final Metadata before updating node:", metadata)
        print("Final KG Nodes:", existing_nodes)
        print("Final KG Relations:", existing_relations)

        #one question unable to find answer to what if entity_node in existing_nodes.append(entity_node) and obj_node/subj_node in existing_nodes.extend([subj_node, obj_node]) is same
        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        print("Node metadata after update:", node.metadata)

        return node

    async def acall(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """Extract triples from nodes async."""
        jobs = [] # jobs to store tasks (coroutines) that will be run concurrently.
        for node in nodes:
            jobs.append(self._aextract(node))

        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting paths from text",
        )


In [ ]:
import re
from llama_index.core.graph_stores import SimplePropertyGraphStore
import networkx as nx
import community.community_louvain as community_louvain
  # Import Louvain module

from llama_index.core.llms import ChatMessage

#Declares GraphRAGStore as a subclass of SimplePropertyGraphStore, inheriting its properties and methods.
class GraphRAGStore(SimplePropertyGraphStore):
    community_summary = {} #empty dictionary to store summaries for different graph communities.
    max_cluster_size = 5

    def generate_community_summary(self, text): #text is the input string containing relationships to summarize.
        """Generate summary for a given text using an LLM."""
        messages = [ #A list of ChatMessage objects. The first message provides instructions to the language model, and the second message contains the text to summarize.
            ChatMessage(
                role="system",
                content=(
                    "You are provided with a set of relationships from a knowledge graph, each represented as "
                    "entity1->entity2->relation->relationship_description. Your task is to create a summary of these "
                    "relationships. The summary should include the names of the entities involved and a concise synthesis "
                    "of the relationship descriptions. The goal is to capture the most critical and relevant details that "
                    "highlight the nature and significance of each relationship. Ensure that the summary is coherent and "
                    "integrates the information in a way that emphasizes the key aspects of the relationships."
                ),
            ),
            ChatMessage(role="user", content=text),
        ]
        response = OpenAI().chat(messages)
        clean_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return clean_response

    def build_communities(self):
        """Builds communities from the graph and summarizes them."""
        nx_graph = self._create_nx_graph() #helper method to convert the internal graph representation into a NetworkX graph.
        partition = community_louvain.best_partition(
            nx_graph
        )
        print(f"Detected clusters: {partition}")  # Print detected clusters

        community_info = self._collect_community_info( # Collects information about the detected communities using a helper method.
            nx_graph, partition
        )
        print(f"Community info collected: {community_info}")  # Print community info

        self._summarize_communities(community_info)

    def _create_nx_graph(self):
        """Converts internal graph representation to NetworkX graph."""
        nx_graph = nx.Graph()  #Creates an empty NetworkX graph.
        for node in self.graph.nodes.values(): #Iterates over all nodes in the internal graph representation.
            nx_graph.add_node(str(node)) #Adds each node to the NetworkX graph.
        for relation in self.graph.relations.values(): #Iterates over all relationships in the graph.
            nx_graph.add_edge( #Adds an edge between two nodes with metadata like relationship and description.
                relation.source_id,
                relation.target_id,
                relationship=relation.label,
                description=relation.properties["relationship_description"],
            )
        return nx_graph

    def _collect_community_info(self, nx_graph, partition):
          """Collect detailed information for each node based on their community."""
          community_info = {}  # Initialize an empty dictionary to hold community-specific relationship details.

          # Populate community_info with node relationships grouped by cluster ID
          for node, cluster_id in partition.items():
              if cluster_id not in community_info:
                  community_info[cluster_id] = []

              for neighbor in nx_graph.neighbors(node):
                  if partition[neighbor] == cluster_id:  # Only include relationships within the same community.
                      edge_data = nx_graph.get_edge_data(node, neighbor)
                      if edge_data:
                          detail = f"{node} -> {neighbor} -> {edge_data['relationship']} -> {edge_data['description']}"
                          community_info[cluster_id].append(detail)

          # Filter out empty communities
          filtered_community_info = {k: v for k, v in community_info.items() if v}

          # Re-index the communities to have sequential numbering
          reindexed_community_info = {}
          for new_id, (old_id, details) in enumerate(filtered_community_info.items()):
              reindexed_community_info[new_id] = details

          return reindexed_community_info

    def _summarize_communities(self, community_info):
        """Generate and store summaries for each community."""
        for community_id, details in community_info.items():
            details_text = (
                "\n".join(details) + "."
            )  # Ensure it ends with a period
            print(f"Summarizing community {community_id} with details: {details_text}")  # Print details before summarizing

            self.community_summary[
                community_id
            ] = self.generate_community_summary(details_text)
            print(f"Summary for community {community_id}: {self.community_summary[community_id]}")  # Print the summary generated for each community

    def get_community_summaries(self):
        """Returns the community summaries, building them if not already done."""
        if not self.community_summary:
            self.build_communities()
        print(f"Final community summaries: {self.community_summary}")  # Print final summaries
        return self.community_summary

In [ ]:
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.llms import LLM


class GraphRAGQueryEngine(CustomQueryEngine):
    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query_str: str) -> str:
        """Process all community summaries to generate answers to a specific query."""
        community_summaries = self.graph_store.get_community_summaries() #Fetches summaries of all communities from GraphRAGStore.
        print(f"Community summaries retrieved: {community_summaries}")  # Print community summaries

        community_answers = [
            self.generate_answer_from_summary(community_summary, query_str)
            for _, community_summary in community_summaries.items()
        ] #Loops through each community summary and uses generate_answer_from_summary to create a response specific to the query for each summary.

        print(f"Individual answers from community summaries: {community_answers}")  # Print individual answers

        final_answer = self.aggregate_answers(community_answers) #Combines all individual answers into one coherent final answer using the aggregate_answers method.
        return final_answer

    def generate_answer_from_summary(self, community_summary, query): #Basic prompting, Generates an answer to the given query using the LLM, based on a community summary.
        """Generate an answer from a community summary based on a given query using LLM."""
        prompt = (
            f"Given the community summary: {community_summary}, "
            f"how would you answer the following query? Query: {query}"
        )
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content="I need an answer based on the above information.",
            ),
        ]
        response = self.llm.chat(messages)
        cleaned_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return cleaned_response

    def aggregate_answers(self, community_answers):#Basic prompting, Combines multiple answers from different community summaries into one cohesive response using the LLM.
        """Aggregate individual community answers into a final, coherent response."""
        # intermediate_text = " ".join(community_answers)
        prompt = "Combine the following intermediate answers into a final, concise response."
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content=f"Intermediate answers: {community_answers}",
            ),
        ]
        final_response = self.llm.chat(messages)
        cleaned_final_response = re.sub(
            r"^assistant:\s*", "", str(final_response)
        ).strip()
        return cleaned_final_response

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=20,
)
nodes = splitter.get_nodes_from_documents(documents)
print(type(nodes))
print(len(nodes))

for i in range(min(5, len(nodes))):
    print(f"Node {i}: {nodes[i]}")

<class 'list'>
5
Node 0: Node ID: 24bef3d2-20c6-49c9-af90-8931cf3f1d00
Text: Chevron: Best Of Breed: JHVEPhoto Like many companies in the O&G
sector, the stock of Chevron (NYSE:CVX) has declined about 10% over
the past 90-days despite the fact that Q2 consensus earnings estimates
have risen sharply (~25%) during that same time frame. Over the years,
Chevron has kept a very strong balance sheet. That allowed the...
Node 1: Node ID: e1e41483-c640-4927-a618-ee4439e8f872
Text: FirstEnergy (NYSE:FE) Posts Earnings Results: FirstEnergy
(NYSE:FE – Get Rating) posted its earnings results on Tuesday. The
utilities provider reported $0.53 earnings per share for the quarter,
topping the consensus estimate of $0.52 by $0.01, RTT News reports.
FirstEnergy had a net margin of 10.85% and a return on equity of
17.17%. During the ...
Node 2: Node ID: c88d33ed-aed1-4032-8448-7a2976a0dc77
Text: Dáil almost suspended after Sinn Féin TD put pager in front of
Minister during firefighters debate: The Dáil wa

In [ ]:
KG_TRIPLET_EXTRACT_TMPL = """
-Goal-
Given a text document, identify all entities and their entity types from the text and all relationships among the identified entities.
Given the text, extract up to {max_knowledge_triplets} entity-relation triplets.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: Type of the entity
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"$$$$<entity_name>$$$$<entity_type>$$$$<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relation: relationship between source_entity and target_entity
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other

Format each relationship as ("relationship"$$$$<source_entity>$$$$<target_entity>$$$$<relation>$$$$<relationship_description>)

3. When finished, output.

-Real Data-
######################
text: {text}
######################
output:"""

In [ ]:
entity_pattern = r'\("entity"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
relationship_pattern = r'\("relationship"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'


def parse_fn(response_str: str) -> Any:
    entities = re.findall(entity_pattern, response_str)
    relationships = re.findall(relationship_pattern, response_str)
    return entities, relationships


kg_extractor = GraphRAGExtractor(
    llm=llm,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=2,
    parse_fn=parse_fn,
)


In [ ]:
from llama_index.core import PropertyGraphIndex

index = PropertyGraphIndex(
    nodes=nodes,
    property_graph_store=GraphRAGStore(),
    kg_extractors=[kg_extractor],
    show_progress=True,
)

# Printing nodes and relationships with descriptive messages

print("This is the full list of nodes:")
print(index.property_graph_store.graph.nodes)

print("This is the length of the full list of nodes:")
print(len(index.property_graph_store.graph.nodes))

# print("\nThis is the full list of relationships:")
# print(index.property_graph_store.graph.relations)

print("\nThese are the values of nodes:")
print(index.property_graph_store.graph.nodes.values())

# print("\nThese are the values of relationships:")
# print(index.property_graph_store.graph.relations.values())

print("\nThis is the first node value:")
print(list(index.property_graph_store.graph.nodes.values())[0])

# print("\nThis is the first relationship value:")
# print(list(index.property_graph_store.graph.relations.values())[0])

# print("\nThis is the 'relationship_description' of the first relationship:")
# print(list(index.property_graph_store.graph.relations.values())[0].properties["relationship_description"])



Extracting paths from text:   0%|          | 0/5 [00:00<?, ?it/s]

Text from node: Chevron: Best Of Breed: JHVEPhoto Like many companies in the O&G sector, the stock of Chevron (NYSE:CVX) has declined about 10% over the past 90-days despite the fact that Q2 consensus earnings estimates have risen sharply (~25%) during that same time frame. Over the years, Chevron has kept a very strong balance sheet. That allowed the...
Initial metadata: {}
Text from node: FirstEnergy (NYSE:FE) Posts Earnings Results: FirstEnergy (NYSE:FE – Get Rating) posted its earnings results on Tuesday. The utilities provider reported $0.53 earnings per share for the quarter, topping the consensus estimate of $0.52 by $0.01, RTT News reports. FirstEnergy had a net margin of 10.85% and a return on equity of 17.17%. During the same period...
If the content contained herein violates any of your rights, including those of copyright, you are requested to immediately notify us using via the following email address operanews-external(at)opera.com
Top News
Initial metadata: {}
Text from 

Extracting paths from text:  20%|██        | 1/5 [00:05<00:22,  5.61s/it]

Extracted Entities: [('FirstEnergy', 'Company', 'FirstEnergy is a utilities provider that recently posted its earnings results on Tuesday. It reported $0.53 earnings per share for the quarter, topping the consensus estimate of $0.52 by $0.01. FirstEnergy had a net margin of 10.85% and a return on equity of 17.17%.'), ('RTT News', 'News Agency', 'RTT News is a news agency that reports on various events, including financial results of companies like FirstEnergy.')]
Extracted Relationships: [('FirstEnergy', 'RTT News', 'Reported by', "FirstEnergy's earnings results were reported by RTT News.")]
Existing KG Nodes: []
Existing KG Relations: []
Metadata copied for entity modification: {}
Metadata after entity modification: {'entity_description': 'RTT News is a news agency that reports on various events, including financial results of companies like FirstEnergy.'}
KG Nodes after adding entities: [EntityNode(label='Company', embedding=None, properties={'entity_description': 'FirstEnergy is a u

Extracting paths from text:  40%|████      | 2/5 [00:09<00:13,  4.59s/it]

...

Extracted Entities: [('Dáil', 'Organization', 'The Dáil is the lower house, and principal chamber, of the Oireachtas (Irish legislature), which also includes the President of Ireland and the Seanad (upper house). It is directly elected at least once in every five years under the system of proportional representation by means of the single transferable vote (STV). Its powers are similar to those of lower houses under many other bicameral parliamentary systems and it is by far the dominant branch of the Oireachtas.'), ('Sinn Féin TD John Brady', 'Person', 'John Brady is a Sinn Féin TD (Teachta Dála) who represents the Wicklow constituency. He is known for his active participation in the Dáil debates.'), ('Minister for Housing Darragh O’Brien', 'Person', 'Darragh O’Brien is the Minister for Housing in Ireland. He is responsible for housing policies and issues.'), ('Retained Firefighters', 'Occupation', 'Retained Firefighters are part-time workers who keep the services going outside of Ire

Extracting paths from text: 100%|██████████| 5/5 [00:25<00:00,  5.01s/it]


Extracted Entities: [('European Commission', 'Organization', 'The European Commission is a branch of the European Union responsible for proposing legislation, implementing decisions, upholding the EU treaties and managing the day-to-day business of the EU.'), ('Huawei Technologies Co.', 'Company', 'Huawei Technologies Co. is a Chinese multinational technology company that provides telecommunications equipment and sells consumer electronics, including smartphones.'), ('ZTE Corp.', 'Company', 'ZTE Corporation is a Chinese multinational telecommunications equipment and systems company.'), ('US', 'Country', 'The United States is a country in North America.'), ('China', 'Country', 'China is a country in East Asia.'), ('TikTok Inc.', 'Company', 'TikTok Inc. is a Chinese multinational internet technology company that provides a social media platform.'), ('5G mobile networks', 'Technology', '5G is the fifth generation technology standard for broadband cellular networks, which cellular phone co

Generating embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]

This is the full list of nodes:
{'24bef3d2-20c6-49c9-af90-8931cf3f1d00': ChunkNode(label='text_chunk', embedding=[-0.018518535420298576, -0.012933580204844475, 0.013154039159417152, -0.008203738369047642, -0.009640060365200043, 0.0076425704173743725, -0.023889712989330292, -0.01448347233235836, -0.0051640793681144714, -0.027844609692692757, -0.008684739470481873, 0.020549429580569267, 0.006035893689841032, 0.01479077897965908, 0.004950301256030798, 0.01421624980866909, 0.011370327323675156, -0.010962813161313534, 0.021083874627947807, -0.03706379234790802, -0.025346076115965843, 0.00760248675942421, -0.02506549283862114, -0.024971965700387955, 0.00024655472952872515, 0.0023064662236720324, 0.014349861070513725, -0.0011407070560380816, 0.002152813132852316, -0.008096848614513874, 0.004161993972957134, -0.0031682595144957304, -0.017062172293663025, 0.0026555259246379137, -0.00988056045025587, -0.014443389140069485, 0.0030129363294690847, -0.017262589186429977, 0.044839974492788315, -0.01

In [ ]:
list(index.property_graph_store.graph.nodes.values())[-1]

EntityNode(label='entity', embedding=None, properties={'relationship_description': 'The European Commission is planning to ban equipment from ZTE Corp. from its own internal telecommunications networks.', 'triplet_source_id': '7a24a532-78d7-4387-890b-8b81cab8a3da'}, name='Deteriorating Relationship')

In [ ]:
list(index.property_graph_store.graph.relations.values())[0]

Relation(label='NYSE:CVX', source_id='Chevron', target_id='has', properties={'relationship_description': 'Chevron has the stock NYSE:CVX, which has declined about 10% over the past 90 days despite the fact that Q2 consensus earnings estimates have risen sharply (~25%) during that same time frame.', 'triplet_source_id': '24bef3d2-20c6-49c9-af90-8931cf3f1d00'})

In [ ]:
list(index.property_graph_store.graph.relations.values())[0].properties[
    "relationship_description"
]

'Chevron has the stock NYSE:CVX, which has declined about 10% over the past 90 days despite the fact that Q2 consensus earnings estimates have risen sharply (~25%) during that same time frame.'

In [ ]:
index.property_graph_store.build_communities()

Detected clusters: {'Chevron: Best Of Breed: JHVEPhoto Like many companies in the O&G sector, the stock of Chevron (NYSE:CVX) has declined about 10% over the past 90-days despite the fact that Q2 consensus earnings estimates have risen sharply (~25%) during that same time frame. Over the years, Chevron has kept a very strong balance sheet. That allowed the...': 0, 'FirstEnergy (NYSE:FE) Posts Earnings Results: FirstEnergy (NYSE:FE – Get Rating) posted its earnings results on Tuesday. The utilities provider reported $0.53 earnings per share for the quarter, topping the consensus estimate of $0.52 by $0.01, RTT News reports. FirstEnergy had a net margin of 10.85% and a return on equity of 17.17%. During the same period...\nIf the content contained herein violates any of your rights, including those of copyright, you are requested to immediately notify us using via the following email address operanews-external(at)opera.com\nTop News': 1, 'Dáil almost suspended after Sinn Féin TD put page

In [ ]:
query_engine = GraphRAGQueryEngine(
    graph_store=index.property_graph_store, llm=llm
)

In [ ]:
response = query_engine.query(
    "What are the main news discussed in the document?"
)
display(Markdown(f"{response.response}"))

Final community summaries: {0: "Chevron and NYSE:CVX are intricately linked through the stock ownership relationship. Despite Chevron's ownership of NYSE:CVX, the stock has experienced a significant decline of approximately 10% over the last 90 days. This decline is notable considering the sharp rise of around 25% in Q2 consensus earnings estimates during the same period. This juxtaposition highlights a disconnect between the stock performance and the positive market outlook based on earnings estimates.", 1: "FirstEnergy's earnings results were reported by RTT News, establishing a relationship where RTT News serves as the reporting entity for FirstEnergy's financial performance.", 2: 'The relationship involves Sinn Féin TD John Brady and Minister for Housing Darragh O’Brien. During a debate on retained firefighters, John Brady engaged in a theatrical act by walking across the chamber and placing an on-call pager in front of Darragh O’Brien. This action was considered a conflict and was

The documents discuss a range of news topics. One highlights the significant decline of approximately 10% in Chevron's stock over the last 90 days, despite a sharp rise in Q2 consensus earnings estimates. Another document reports on FirstEnergy's earnings results. In political news, Sinn Féin TD John Brady made a theatrical gesture during a debate on retained firefighters, escalating the ongoing dispute over pay and working conditions between the firefighters and the Minister for Housing, Darragh O’Brien. In technology, Epic has developed the MetaHuman Animator, a tool that can create hyperrealistic animations, showcased in the short film "Blue Dot" starring Radivoje Bukvić. Lastly, the European Commission has banned the use of TikTok by its staff due to security concerns, reflecting the deteriorating relationship between the US and China and measures to block Chinese technology from critical telecommunications networks.

In [ ]:
response = query_engine.query("What are news related to financial sector?")
display(Markdown(f"{response.response}"))

Final community summaries: {0: "Chevron and NYSE:CVX are intricately linked through the stock ownership relationship. Despite Chevron's ownership of NYSE:CVX, the stock has experienced a significant decline of approximately 10% over the last 90 days. This decline is notable considering the sharp rise of around 25% in Q2 consensus earnings estimates during the same period. This juxtaposition highlights a disconnect between the stock performance and the positive market outlook based on earnings estimates.", 1: "FirstEnergy's earnings results were reported by RTT News, establishing a relationship where RTT News serves as the reporting entity for FirstEnergy's financial performance.", 2: 'The relationship involves Sinn Féin TD John Brady and Minister for Housing Darragh O’Brien. During a debate on retained firefighters, John Brady engaged in a theatrical act by walking across the chamber and placing an on-call pager in front of Darragh O’Brien. This action was considered a conflict and was

The provided information, summaries, and community summaries do not contain any news related to the financial sector.